In [1]:
# STEP 1: CHECK ALL CLEANED DATASET FILES

import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"

csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

print("Total CSV files:", len(csv_files))

for file in csv_files:

    df_check = pd.read_csv(file, nrows=5)

    print("\nFile:", file.name)
    print("Columns:", len(df_check.columns))
    print("Label column available:", "Label" in df_check.columns)

    if "Label" in df_check.columns:
        print("Sample labels:")
        print(df_check["Label"].value_counts())

Total CSV files: 8

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Friday-WorkingHours-Morning.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Monday-WorkingHours.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Columns: 79
Label column available: True
Sample labels:
Label
BENIGN    5
Name: count, dtype: int64

File: Tuesday-WorkingHours.pcap_ISCX.csv
Colu

In [2]:
# STEP 2: COMPLETE LABEL DISTRIBUTION

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"

csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

all_label_counts = {}

for file in csv_files:

    print(f"\nProcessing: {file.name}")

    label_counts = {}

    # Read large CSV files in smaller chunks
    for chunk in pd.read_csv(
        file,
        usecols=["Label"],
        chunksize=100000
    ):

        labels = (
            chunk["Label"]
            .astype(str)
            .str.strip()
        )

        counts = labels.value_counts()

        for label, count in counts.items():

            label_counts[label] = (
                label_counts.get(label, 0) + count
            )

    all_label_counts[file.name] = label_counts


# Convert results into a DataFrame

label_summary = pd.DataFrame(
    all_label_counts
).fillna(0).astype(int).T

label_summary["Total Records"] = (
    label_summary.sum(axis=1)
)

print("\nCOMPLETE LABEL DISTRIBUTION")

display(label_summary)

print("\nOVERALL LABEL COUNTS")

overall_counts = label_summary.drop(
    columns=["Total Records"]
).sum()

display(overall_counts)

print("\nTOTAL RECORDS:", overall_counts.sum())


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv

Processing: Monday-WorkingHours.pcap_ISCX.csv

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

Processing: Tuesday-WorkingHours.pcap_ISCX.csv

Processing: Wednesday-workingHours.pcap_ISCX.csv

COMPLETE LABEL DISTRIBUTION


,DDoS,BENIGN,PortScan,Bot,Infiltration,Web Attack � Brute Force,Web Attack � XSS,Web Attack � Sql Injection,FTP-Patator,SSH-Patator,DoS Hulk,DoS slowloris,DoS Slowhttptest,DoS GoldenEye,Heartbleed,Total Records
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,128021,95854,0,0,0,0,0,0,0,0,0,0,0,0,0,223875
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,0,124040,114652,0,0,0,0,0,0,0,0,0,0,0,0,238692
Friday-WorkingHours-Morning.pcap_ISCX.csv,0,183673,0,1956,0,0,0,0,0,0,0,0,0,0,0,185629
Monday-WorkingHours.pcap_ISCX.csv,0,512438,0,0,0,0,0,0,0,0,0,0,0,0,0,512438
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,0,260462,0,0,36,0,0,0,0,0,0,0,0,0,0,260498
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,0,163394,0,0,0,1470,652,21,0,0,0,0,0,0,0,165537
Tuesday-WorkingHours.pcap_ISCX.csv,0,419782,0,0,0,0,0,0,5935,3545,0,0,0,0,0,429262
Wednesday-workingHours.pcap_ISCX.csv,0,425495,0,0,0,0,0,0,0,0,174334,5385,5228,10286,11,620739



OVERALL LABEL COUNTS


DDoS                           128021
BENIGN                        2185138
PortScan                       114652
Bot                              1956
Infiltration                       36
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Web Attack � Sql Injection         21
FTP-Patator                      5935
SSH-Patator                      3545
DoS Hulk                       174334
DoS slowloris                    5385
DoS Slowhttptest                 5228
DoS GoldenEye                   10286
Heartbleed                         11
dtype: int64


TOTAL RECORDS: 2636670


In [3]:
# STEP 3: ATTACK DISTRIBUTION BY DATASET

import pandas as pd

# Use label counts calculated in Step 2
distribution = pd.DataFrame.from_dict(
    all_label_counts,
    orient="index"
).fillna(0).astype(int)

# Calculate total records
distribution["Total Records"] = distribution.sum(axis=1)

# Calculate total attacks (all labels except BENIGN)
attack_columns = [
    col for col in distribution.columns
    if col != "BENIGN" and col != "Total Records"
]

distribution["ATTACK Total"] = (
    distribution[attack_columns].sum(axis=1)
)

# Calculate benign records
distribution["BENIGN Total"] = distribution["BENIGN"]

# Calculate attack percentage
distribution["Attack Percentage"] = (
    distribution["ATTACK Total"]
    / distribution["Total Records"] * 100
).round(2)

# Display useful columns
display(
    distribution[
        [
            "BENIGN Total",
            "ATTACK Total",
            "Total Records",
            "Attack Percentage"
        ]
    ].sort_index()
)

# Display attack families by dataset
print("\nATTACK FAMILY DISTRIBUTION BY DATASET")

display(
    distribution[attack_columns]
)

,BENIGN Total,ATTACK Total,Total Records,Attack Percentage
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,95854,128021,223875,57.18
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,124040,114652,238692,48.03
Friday-WorkingHours-Morning.pcap_ISCX.csv,183673,1956,185629,1.05
Monday-WorkingHours.pcap_ISCX.csv,512438,0,512438,0.00
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,260462,36,260498,0.01
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,163394,2143,165537,1.29
Tuesday-WorkingHours.pcap_ISCX.csv,419782,9480,429262,2.21
Wednesday-workingHours.pcap_ISCX.csv,425495,195244,620739,31.45



ATTACK FAMILY DISTRIBUTION BY DATASET


,DDoS,PortScan,Bot,Infiltration,Web Attack � Brute Force,Web Attack � XSS,Web Attack � Sql Injection,FTP-Patator,SSH-Patator,DoS Hulk,DoS slowloris,DoS Slowhttptest,DoS GoldenEye,Heartbleed
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,128021,0,0,0,0,0,0,0,0,0,0,0,0,0
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,0,114652,0,0,0,0,0,0,0,0,0,0,0,0
Friday-WorkingHours-Morning.pcap_ISCX.csv,0,0,1956,0,0,0,0,0,0,0,0,0,0,0
Monday-WorkingHours.pcap_ISCX.csv,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,0,0,0,36,0,0,0,0,0,0,0,0,0,0
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,0,0,0,0,1470,652,21,0,0,0,0,0,0,0
Tuesday-WorkingHours.pcap_ISCX.csv,0,0,0,0,0,0,0,5935,3545,0,0,0,0,0
Wednesday-workingHours.pcap_ISCX.csv,0,0,0,0,0,0,0,0,0,174334,5385,5228,10286,11


In [4]:
# STEP 4: DAY-WISE DATA PREPARATION

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"

# Define capture-day groups

train_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv"
]

validation_files = [
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
]

test_files = [
    "Wednesday-workingHours.pcap_ISCX.csv"
]


# Function to load selected datasets

def load_day_datasets(file_list, split_name):

    frames = []

    for filename in file_list:

        file_path = PROCESSED_DIR / filename

        if not file_path.exists():
            raise FileNotFoundError(file_path)

        df = pd.read_csv(file_path)

        # Add capture-day information
        df["Capture_Day"] = split_name

        frames.append(df)

        print(
            filename,
            "->",
            f"{len(df):,} records"
        )

    combined = pd.concat(
        frames,
        ignore_index=True
    )

    return combined


# Load training data
train_df = load_day_datasets(
    train_files,
    "TRAIN"
)

# Load validation data
validation_df = load_day_datasets(
    validation_files,
    "VALIDATION"
)

# Load final test data
test_df = load_day_datasets(
    test_files,
    "TEST"
)


# Display dataset sizes

print("\nDATASET SPLIT SUMMARY")

print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Testing:", test_df.shape)


# Convert labels into binary format

def convert_labels(df):

    df = df.copy()

    df["Label"] = (
        df["Label"]
        .astype(str)
        .str.strip()
        .str.upper()
        .ne("BENIGN")
        .astype(int)
    )

    return df


train_df = convert_labels(train_df)

validation_df = convert_labels(validation_df)

test_df = convert_labels(test_df)


# Check class distributions

print("\nTRAINING LABEL DISTRIBUTION")
print(train_df["Label"].value_counts())

print("\nVALIDATION LABEL DISTRIBUTION")
print(validation_df["Label"].value_counts())

print("\nTEST LABEL DISTRIBUTION")
print(test_df["Label"].value_counts())

Monday-WorkingHours.pcap_ISCX.csv -> 512,438 records
Tuesday-WorkingHours.pcap_ISCX.csv -> 429,262 records
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv -> 223,875 records
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv -> 238,692 records
Friday-WorkingHours-Morning.pcap_ISCX.csv -> 185,629 records
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> 260,498 records
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv -> 165,537 records
Wednesday-workingHours.pcap_ISCX.csv -> 620,739 records

DATASET SPLIT SUMMARY
Training: (1589896, 80)
Validation: (426035, 80)
Testing: (620739, 80)

TRAINING LABEL DISTRIBUTION
Label
0    1335787
1     254109
Name: count, dtype: int64

VALIDATION LABEL DISTRIBUTION
Label
0    423856
1      2179
Name: count, dtype: int64

TEST LABEL DISTRIBUTION
Label
0    425495
1    195244
Name: count, dtype: int64


In [5]:
# STEP 5: FEATURE VALIDATION

# Remove target and metadata columns from model inputs
X_train = train_df.drop(columns=["Label", "Capture_Day"])
X_val = validation_df.drop(columns=["Label", "Capture_Day"])
X_test = test_df.drop(columns=["Label", "Capture_Day"])

# Separate target variables
y_train = train_df["Label"]
y_val = validation_df["Label"]
y_test = test_df["Label"]

# Check feature counts
print("TRAINING FEATURES:", X_train.shape)
print("VALIDATION FEATURES:", X_val.shape)
print("TESTING FEATURES:", X_test.shape)

# Check whether feature columns match
print("\nFEATURE COLUMNS MATCH:")
print(
    list(X_train.columns) == list(X_val.columns)
    == list(X_test.columns)
)

# Check missing values
print("\nMISSING VALUES:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Testing:", X_test.isna().sum().sum())

# Check infinite values
print("\nINFINITE VALUES:")
print("Training:", np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())
print("Validation:", np.isinf(X_val.select_dtypes(include=np.number)).sum().sum())
print("Testing:", np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())

TRAINING FEATURES: (1589896, 78)
VALIDATION FEATURES: (426035, 78)
TESTING FEATURES: (620739, 78)

FEATURE COLUMNS MATCH:
True

MISSING VALUES:
Training: 2096
Validation: 628
Testing: 648

INFINITE VALUES:
Training: 0
Validation: 0
Testing: 0


In [6]:
# STEP 6: HANDLE MISSING VALUES

from sklearn.impute import SimpleImputer

# Create imputer
imputer = SimpleImputer(strategy="median")

# Learn median values ONLY from training data
X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Apply the same medians to validation and testing
X_val = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Verify missing values are handled
print("MISSING VALUES AFTER IMPUTATION")

print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Testing:", X_test.isna().sum().sum())

print("\nFINAL DATASET SHAPES")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

MISSING VALUES AFTER IMPUTATION
Training: 0
Validation: 0
Testing: 0

FINAL DATASET SHAPES
Training: (1589896, 78)
Validation: (426035, 78)
Testing: (620739, 78)


In [7]:
# STEP 7: CHECK CONSTANT FEATURES

constant_features = [
    col for col in X_train.columns
    if X_train[col].nunique() <= 1
]

print("Number of constant features:", len(constant_features))
print("\nConstant features:")
print(constant_features)

# Remove constant features from all datasets
X_train = X_train.drop(columns=constant_features)
X_val = X_val.drop(columns=constant_features)
X_test = X_test.drop(columns=constant_features)

print("\nFEATURES AFTER REMOVING CONSTANT FEATURES")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Number of constant features: 10

Constant features:
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

FEATURES AFTER REMOVING CONSTANT FEATURES
Training: (1589896, 68)
Validation: (426035, 68)
Testing: (620739, 68)


In [8]:
# STEP 8: CHECK CLASS IMBALANCE

import pandas as pd

print("TRAINING CLASS DISTRIBUTION")
print(y_train.value_counts())

print("\nTRAINING CLASS PERCENTAGES")
print((y_train.value_counts(normalize=True) * 100).round(2))

# Calculate imbalance ratio
benign_count = (y_train == 0).sum()
attack_count = (y_train == 1).sum()

imbalance_ratio = benign_count / attack_count

print("\nCLASS IMBALANCE RATIO:")
print(round(imbalance_ratio, 2))

print("\nVALIDATION CLASS DISTRIBUTION")
print(y_val.value_counts())

print("\nTEST CLASS DISTRIBUTION")
print(y_test.value_counts())

TRAINING CLASS DISTRIBUTION
Label
0    1335787
1     254109
Name: count, dtype: int64

TRAINING CLASS PERCENTAGES
Label
0    84.02
1    15.98
Name: proportion, dtype: float64

CLASS IMBALANCE RATIO:
5.26

VALIDATION CLASS DISTRIBUTION
Label
0    423856
1      2179
Name: count, dtype: int64

TEST CLASS DISTRIBUTION
Label
0    425495
1    195244
Name: count, dtype: int64


In [9]:
# STEP 9: RANDOM FOREST MODEL COMPARISON

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import time

rf_results = []

rf_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "Random Forest Balanced": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

for model_name, model in rf_models.items():

    print(f"\nTraining {model_name}...")

    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    # Predict on validation data
    y_val_pred = model.predict(X_val)

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    rf_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (seconds)": round(training_time, 2)
    })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, y_val_pred))

rf_comparison = pd.DataFrame(rf_results)

print("\nRANDOM FOREST COMPARISON")
display(rf_comparison)


Training Random Forest...
Accuracy: 0.9923
Precision: 0.0406
Recall: 0.0225
F1 Score: 0.0289
Training Time: 252.05 seconds

Confusion Matrix:
[[422697   1159]
 [  2130     49]]

Training Random Forest Balanced...
Accuracy: 0.9216
Precision: 0.0170
Recall: 0.2515
F1 Score: 0.0318
Training Time: 304.97 seconds

Confusion Matrix:
[[392091  31765]
 [  1631    548]]

RANDOM FOREST COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,Training Time (seconds)
0,Random Forest,0.992280,0.040563,0.022487,0.028934,252.05
1,Random Forest Balanced,0.921612,0.016959,0.251492,0.031775,304.97


In [10]:
# STEP 10: EXTRA TREES MODEL COMPARISON

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import time

et_results = []

et_models = {
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight=None,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees Balanced": ExtraTreesClassifier(
        n_estimators=200,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

for model_name, model in et_models.items():

    print(f"\nTraining {model_name}...")

    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_val_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    et_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (seconds)": round(training_time, 2)
    })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, y_val_pred))

et_comparison = pd.DataFrame(et_results)

print("\nEXTRA TREES COMPARISON")
display(et_comparison)


Training Extra Trees...
Accuracy: 0.8937
Precision: 0.0006
Recall: 0.0110
F1 Score: 0.0011
Training Time: 138.97 seconds

Confusion Matrix:
[[380720  43136]
 [  2155     24]]

Training Extra Trees Balanced...
Accuracy: 0.8938
Precision: 0.0018
Recall: 0.0363
F1 Score: 0.0035
Training Time: 164.95 seconds

Confusion Matrix:
[[380711  43145]
 [  2100     79]]

EXTRA TREES COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,Training Time (seconds)
0,Extra Trees,0.893692,0.000556,0.011014,0.001059,138.97
1,Extra Trees Balanced,0.893800,0.001828,0.036255,0.003480,164.95


In [11]:
# STEP 11: VALIDATION ATTACK FAMILY ANALYSIS

# Predict using the balanced Random Forest
rf_balanced = rf_models["Random Forest Balanced"]

y_val_pred = rf_balanced.predict(X_val)

# Keep original attack labels
validation_analysis = validation_df[["Label"]].copy()

validation_analysis["Actual_Label"] = validation_df["Label"].astype(str)
validation_analysis["Predicted"] = y_val_pred

# Convert original labels into BENIGN or ATTACK
validation_analysis["Actual_Binary"] = (
    validation_analysis["Actual_Label"]
    .str.strip()
    .str.upper()
    .ne("BENIGN")
    .astype(int)
)

# Analyze each attack family
attack_analysis = validation_analysis[
    validation_analysis["Actual_Binary"] == 1
].groupby("Actual_Label").agg(
    Total_Records=("Predicted", "count"),
    Detected=("Predicted", "sum")
)

attack_analysis["Missed"] = (
    attack_analysis["Total_Records"] - attack_analysis["Detected"]
)

attack_analysis["Recall_Percent"] = (
    attack_analysis["Detected"] /
    attack_analysis["Total_Records"] * 100
).round(2)

display(attack_analysis.sort_values("Recall_Percent", ascending=False))

,Total_Records,Detected,Missed,Recall_Percent
Actual_Label,,,,
1,2179,548,1631,25.15
0,423856,31765,392091,7.49


In [12]:
# STEP 11: RECOVER ORIGINAL ATTACK LABELS

# Read only the original Label column
original_labels = []

for filename in validation_files:
    file_path = PROCESSED_DIR / filename

    labels = pd.read_csv(file_path, usecols=["Label"])
    original_labels.append(labels)

original_validation_labels = pd.concat(
    original_labels, ignore_index=True
)["Label"].astype(str).str.strip()

# Check row alignment
print("Validation rows:", len(y_val_pred))
print("Original label rows:", len(original_validation_labels))

# Create attack-family analysis
attack_analysis_df = pd.DataFrame({
    "Actual_Label": original_validation_labels,
    "Predicted": y_val_pred
})

# Keep only actual attacks
attack_analysis_df = attack_analysis_df[
    attack_analysis_df["Actual_Label"].str.upper() != "BENIGN"
]

# Calculate detection results for each attack family
attack_family_results = attack_analysis_df.groupby(
    "Actual_Label"
).agg(
    Total_Records=("Predicted", "count"),
    Detected=("Predicted", "sum")
)

attack_family_results["Missed"] = (
    attack_family_results["Total_Records"]
    - attack_family_results["Detected"]
)

attack_family_results["Recall_Percent"] = (
    attack_family_results["Detected"]
    / attack_family_results["Total_Records"] * 100
).round(2)

display(
    attack_family_results.sort_values(
        "Recall_Percent", ascending=False
    )
)

Validation rows: 426035
Original label rows: 426035


,Total_Records,Detected,Missed,Recall_Percent
Actual_Label,,,,
Web Attack � XSS,652,230,422,35.28
Web Attack � Brute Force,1470,318,1152,21.63
Infiltration,36,0,36,0.00
Web Attack � Sql Injection,21,0,21,0.00


In [13]:
# STEP 12: THRESHOLD COMPARISON

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Get attack probabilities
attack_probabilities = rf_balanced.predict_proba(X_val)[
    :, list(rf_balanced.classes_).index(1)
]

threshold_results = []

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

for threshold in thresholds:

    predictions = (attack_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_val, predictions, zero_division=0
    )

    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    false_positive_rate = fp / (fp + tn)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1 Score": round(f1, 4),
        "False Positives": fp,
        "False Positive Rate": round(false_positive_rate, 4)
    })

threshold_comparison = pd.DataFrame(threshold_results)

display(threshold_comparison)

,Threshold,Precision,Recall,F1 Score,False Positives,False Positive Rate
0,0.30,0.0519,0.8017,0.0974,31935,0.0753
1,0.40,0.0222,0.3318,0.0416,31851,0.0751
2,0.50,0.0170,0.2520,0.0318,31766,0.0749
3,0.60,0.0124,0.1744,0.0232,30266,0.0714
4,0.70,0.0773,0.0399,0.0527,1038,0.0024
5,0.80,0.0169,0.0078,0.0107,990,0.0023
6,0.90,0.0030,0.0014,0.0019,981,0.0023
7,0.95,0.0010,0.0005,0.0006,976,0.0023


In [14]:
# STEP 13: PREPARE DAY-WISE CROSS-VALIDATION

from sklearn.model_selection import GroupKFold

# Capture day for each training record
train_groups = np.concatenate([
    np.full(512438, "Monday"),
    np.full(429262, "Tuesday"),
    np.full(223875 + 238692 + 185629, "Friday")
])

# Verify group information
print("Training records:", len(X_train))
print("Group records:", len(train_groups))

print("\nRecords per capture day:")
print(pd.Series(train_groups).value_counts())

# Create cross-validation strategy
group_cv = GroupKFold(n_splits=3)

print("\nCross-validation folds:", group_cv.get_n_splits())

Training records: 1589896
Group records: 1589896

Records per capture day:
Friday     648196
Monday     512438
Tuesday    429262
Name: count, dtype: int64

Cross-validation folds: 3


In [15]:
# STEP 14: RANDOM FOREST GROUP CROSS-VALIDATION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
import numpy as np
import pandas as pd
import time

cv_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_cv.split(X_train, y_train, groups=train_groups), start=1
):

    print(f"\nStarting Fold {fold}")

    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    print("Training records:", len(train_idx))
    print("Validation records:", len(val_idx))

    model = RandomForestClassifier(
        n_estimators=150,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    start_time = time.time()

    model.fit(X_fold_train, y_fold_train)

    predictions = model.predict(X_fold_val)

    elapsed = time.time() - start_time

    results = {
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, predictions),
        "Precision": precision_score(
            y_fold_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_fold_val, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_fold_val, predictions, zero_division=0
        ),
        "Training Time": round(elapsed, 2)
    }

    cv_results.append(results)

    print("Accuracy:", round(results["Accuracy"], 4))
    print("Precision:", round(results["Precision"], 4))
    print("Recall:", round(results["Recall"], 4))
    print("F1 Score:", round(results["F1 Score"], 4))
    print("Time:", round(elapsed, 2), "seconds")

cv_comparison = pd.DataFrame(cv_results)

print("\nGROUP K-FOLD RESULTS")
display(cv_comparison)

print("\nAVERAGE CROSS-VALIDATION PERFORMANCE")
display(
    cv_comparison.drop(columns=["Fold"]).mean().to_frame("Average")
)


Starting Fold 1
Training records: 941700
Validation records: 648196
Accuracy: 0.6226
Precision: 1.0
Recall: 0.0
F1 Score: 0.0
Time: 81.41 seconds

Starting Fold 2
Training records: 1077458
Validation records: 512438
Accuracy: 0.9997
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Time: 146.88 seconds

Starting Fold 3
Training records: 1160634
Validation records: 429262
Accuracy: 0.9779
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Time: 156.19 seconds

GROUP K-FOLD RESULTS


,Fold,Accuracy,Precision,Recall,F1 Score,Training Time
0,1,0.622605,1.0,0.000012,0.000025,81.41
1,2,0.999739,0.0,0.000000,0.000000,146.88
2,3,0.977916,0.0,0.000000,0.000000,156.19



AVERAGE CROSS-VALIDATION PERFORMANCE


,Average
Accuracy,0.866753
Precision,0.333333
Recall,0.000004
F1 Score,0.000008
Training Time,128.160000


In [17]:
# STEP 15: PREPARE DEVELOPMENT DATA FOR 4-DAY CROSS-VALIDATION

# Combine Monday, Tuesday, Friday and Thursday
development_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

# Separate features and target
X_dev_raw = development_df.drop(
    columns=["Label", "Capture_Day"]
)

y_dev = development_df["Label"]

# Capture day groups
day_groups = development_df["Capture_Day"]

print("Development data shape:", X_dev_raw.shape)

print("\nRecords per capture day:")
print(day_groups.value_counts())

print("\nTarget distribution:")
print(y_dev.value_counts())

print("\nNumber of unique capture days:", day_groups.nunique())

Development data shape: (2015931, 78)

Records per capture day:
Capture_Day
TRAIN         1589896
VALIDATION     426035
Name: count, dtype: int64

Target distribution:
Label
0    1759643
1     256288
Name: count, dtype: int64

Number of unique capture days: 2


In [19]:
# Combine training and validation features

X_dev = pd.concat(
    [X_train, X_val],
    ignore_index=True
)

# Combine training and validation target labels

y_dev = pd.concat(
    [y_train, y_val],
    ignore_index=True
)

print("Development feature shape:", X_dev.shape)
print("Development target shape:", y_dev.shape)

Development feature shape: (2015931, 68)
Development target shape: (2015931,)
